## Notebook Instructions

1. If you are new to Jupyter notebooks, please go through this introductory manual [here](https://quantra.quantinsti.com/quantra-notebook).
2. Any changes made in this notebook would be lost after you close the browser window. **You can download the notebook to save your work on your PC.**
3. Before running this notebook on your local PC:
   - You need to set up a Python environment and the relevant packages on your local PC. To do so, go through the section on **"Run Codes Locally on Your Machine"** in the course.
   - You need to **download the zip file available in the last unit** of this course. The zip file contains the data files and/or Python modules that might be required to run this notebook.


# Trading Signals

In our last notebook, we successfully calculated a Simple Moving Average (SMA). Now, we'll use that analysis to make decisions.

This notebook will guide you through implementing a classic SMA Crossover strategy to generate real-time buy and sell signals. We're building the 'brain' of our trading system.

## Outline

1.  **The Starting Code**: Importing libraries and setting up our WebSocket connector.

2.  **Adding 'State'**: Introducing a variable to remember if the price is above or below the SMA.
3.  **Implementing Crossover Logic**: Modifying the `on_ticks` function to detect the exact moment the price crosses the SMA.
4.  **Generating Live Signals**: Running the code to see our 'BUY' and 'SELL' signals appear in real-time.
5.  **Conclusion & Next Steps**: Recapping and looking forward to trade execution.

### 1. The Starting Code

We’ll import the `KiteTicker` class for WebSocket streaming and the `logging` module to display status messages.


In [1]:
# Import the necessary libraries
from kiteconnect import KiteTicker
import logging

# --- Setup ---
logging.basicConfig(level=logging.INFO)

# --- Your Credentials ---
# IMPORTANT: Replace these with your actual api_key and access_token.
# You get the access_token after a successful login via the Kite Connect API.
api_key = "YOUR_API_KEY"
access_token = "YOUR_ACCESS_TOKEN"


# --- Initialise the Ticker ---
kws = KiteTicker(api_key, access_token)

### 2. Adding 'State' for Memory

To detect a crossover, we can't just check if the price is above the SMA. That would generate a signal on every tick! We only care about the *moment* it crosses.

To do this, we need to give our script 'memory'. We'll create a variable called `position_status` to remember if the price was previously in a 'BULL' state (above the SMA) or a 'BEAR' state (below the SMA).

### 3. Implementing Crossover Logic in `on_ticks`

We will now upgrade our `on_ticks` function. After calculating the SMA, we'll add the core logic of our strategy:

-   **Bullish Crossover**: If the current price is greater than the SMA, AND our `position_status` remembers that it was previously 'BEAR', it's a BUY signal.
-   **Bearish Crossover**: If the current price is less than the SMA, AND our `position_status` remembers that it was previously 'BULL', it's a SELL signal.

Once a signal is generated, we immediately update the `position_status` to prevent repeated signals.

In [2]:
# --- Global variables for storing data and state ---
ltp_history = []
SMA_PERIOD = 10

# This is our 'memory'. It can be None, 'BULL', or 'BEAR'.
position_status = None

# --- Callback Functions (with strategy logic) ---

def on_ticks(ws, ticks):
    # We need to tell the function to modify the global variable, not a local one.
    global position_status
    
    for tick in ticks:
        last_price = tick['last_price']
        ltp_history.append(last_price)
        
        # Ensure we only keep the last SMA_PERIOD prices
        if len(ltp_history) > SMA_PERIOD:
            ltp_history.pop(0)

        # Only proceed if we have enough data for an SMA
        if len(ltp_history) == SMA_PERIOD:
            sma = sum(ltp_history) / SMA_PERIOD
            
            # --- STRATEGY LOGIC ---
            
            # 1. Bullish Crossover (Price crosses ABOVE SMA)
            if last_price > sma and position_status == 'BEAR':
                position_status = 'BULL'
                # Using WARNING level to make the signal stand out in the logs
                logging.warning(f"BULLISH CROSSOVER: BUY SIGNAL! LTP: {last_price:.2f} | SMA: {sma:.2f}")

            # 2. Bearish Crossover (Price crosses BELOW SMA)
            if last_price < sma and position_status == 'BULL':
                position_status = 'BEAR'
                # Using ERROR level to make the signal stand out
                logging.warning(f"BEARISH CROSSOVER: SELL SIGNAL! LTP: {last_price:.2f} | SMA: {sma:.2f}")

            # 3. Set initial status on the very first calculation
            if position_status is None:
                position_status = 'BULL' if last_price > sma else 'BEAR'

            # Log the continuous data stream for monitoring
            logging.info(f"LTP: {last_price:.2f} | SMA({SMA_PERIOD}): {sma:.2f} | Status: {position_status}")

def on_connect(ws, response):
    """This function runs once the connection is established."""
    logging.info("Successfully connected! Waiting for data...")
    instrument_tokens = [738561] # Reliance
    ws.subscribe(instrument_tokens)
    ws.set_mode(ws.MODE_LTP, instrument_tokens)

def on_close(ws, code, reason):
    """This function runs when the connection is closed."""
    logging.info(f"Connection closed: {code} - {reason}")
    ws.stop()

### 4. Generating Live Signals

Now, let's run our complete script. The setup is the same. After connecting, the script will first gather 10 ticks. Then, it will start printing the live LTP and SMA. 

**Watch the logs carefully!** You will see the normal `INFO` messages on every tick, but a colorful `WARNING` (BUY) or (SELL) message will appear only at the exact moment a crossover occurs.

In [3]:
# --- Assign Callbacks and Connect ---
kws.on_ticks = on_ticks
kws.on_connect = on_connect
kws.on_close = on_close

print("Connecting to WebSocket... (This will run indefinitely. Interrupt the kernel to stop.)")
kws.connect()

INFO:root:Successfully connected! Waiting for data...


Connecting to WebSocket... (This will run indefinitely. Interrupt the kernel to stop.)


INFO:root:LTP: 1484.50 | SMA(10): 1484.38 | Status: BULL
INFO:root:LTP: 1484.30 | SMA(10): 1484.38 | Status: BEAR
INFO:root:LTP: 1484.50 | SMA(10): 1484.40 | Status: BULL
INFO:root:LTP: 1484.30 | SMA(10): 1484.40 | Status: BEAR
INFO:root:LTP: 1484.30 | SMA(10): 1484.38 | Status: BEAR
INFO:root:LTP: 1484.30 | SMA(10): 1484.38 | Status: BEAR
INFO:root:LTP: 1484.50 | SMA(10): 1484.38 | Status: BULL
INFO:root:LTP: 1484.30 | SMA(10): 1484.38 | Status: BEAR
INFO:root:LTP: 1484.50 | SMA(10): 1484.40 | Status: BULL
INFO:root:LTP: 1484.50 | SMA(10): 1484.40 | Status: BULL
INFO:root:LTP: 1484.50 | SMA(10): 1484.40 | Status: BULL
INFO:root:LTP: 1484.50 | SMA(10): 1484.42 | Status: BULL
INFO:root:LTP: 1484.60 | SMA(10): 1484.43 | Status: BULL
INFO:root:LTP: 1484.50 | SMA(10): 1484.45 | Status: BULL
INFO:root:LTP: 1484.60 | SMA(10): 1484.48 | Status: BULL
INFO:root:LTP: 1484.60 | SMA(10): 1484.51 | Status: BULL
INFO:root:LTP: 1484.60 | SMA(10): 1484.52 | Status: BULL
INFO:root:LTP: 1484.60 | SMA(10

INFO:root:LTP: 1484.90 | SMA(10): 1484.94 | Status: BEAR
INFO:root:LTP: 1484.90 | SMA(10): 1484.93 | Status: BEAR
INFO:root:LTP: 1484.90 | SMA(10): 1484.92 | Status: BEAR
INFO:root:LTP: 1484.90 | SMA(10): 1484.92 | Status: BEAR
INFO:root:LTP: 1484.90 | SMA(10): 1484.92 | Status: BEAR
INFO:root:LTP: 1484.90 | SMA(10): 1484.92 | Status: BEAR
INFO:root:LTP: 1484.90 | SMA(10): 1484.91 | Status: BEAR
INFO:root:LTP: 1484.80 | SMA(10): 1484.89 | Status: BEAR
INFO:root:LTP: 1484.90 | SMA(10): 1484.89 | Status: BULL
INFO:root:LTP: 1484.90 | SMA(10): 1484.89 | Status: BULL
INFO:root:LTP: 1485.00 | SMA(10): 1484.90 | Status: BULL
INFO:root:LTP: 1485.00 | SMA(10): 1484.91 | Status: BULL
INFO:root:LTP: 1484.90 | SMA(10): 1484.91 | Status: BEAR
INFO:root:LTP: 1485.00 | SMA(10): 1484.92 | Status: BULL
INFO:root:LTP: 1485.00 | SMA(10): 1484.93 | Status: BULL
INFO:root:LTP: 1485.00 | SMA(10): 1484.94 | Status: BULL
INFO:root:LTP: 1485.00 | SMA(10): 1484.95 | Status: BULL
INFO:root:LTP: 1484.90 | SMA(10

### 5. Conclusion & Next Steps

Fantastic! We've now built a complete, albeit simple, trading strategy. We have successfully translated the technical analysis concept of an SMA crossover into concrete trading logic that generates clear, timely signals.

**Recap:**
- We introduced a 'state' variable (`position_status`) to give our script memory.
- We built logic to detect the precise moment of a crossover, not just the state.
- We generated distinct 'BUY' and 'SELL' signals based on this logic.

The system is now complete: **Data Feed → Analysis → Signals**. The only missing piece is execution.

In our next video, we will connect this logic to the Kite Connect order placement functions. We'll take our 'BUY SIGNAL' and turn it into a real buy order sent to the exchange. This is where it all comes together!